In [0]:
CREATE SCHEMA IF NOT EXISTS training_catalog.reject;
USE training_catalog.reject;

In [0]:
CREATE OR REPLACE TABLE rejected_customers
USING DELTA
AS
SELECT
    *,
    CASE
        WHEN CustomerID IS NULL THEN 'Missing CustomerID'
        WHEN CustomerName IS NULL THEN 'Missing CustomerName'
        WHEN Email IS NULL THEN 'Missing Email'
        WHEN City IS NULL THEN 'Missing City'
        WHEN Address IS NULL THEN 'Missing Address'
        WHEN TRIM(CustomerName) NOT RLIKE '^[A-Za-z ]+$'
            THEN 'Invalid CustomerName Format'
        WHEN TRIM(Email) NOT RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'
            THEN 'Invalid Email Format'
        WHEN TRIM(City) NOT RLIKE '^[A-Za-z ]+$'
            THEN 'Invalid City Format'
        WHEN LENGTH(TRIM(Address)) < 5
            THEN 'Invalid Address'
        ELSE 'Invalid Customer Record'
    END AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.customers_raw
WHERE CustomerID IS NULL
   OR CustomerName IS NULL
   OR Email IS NULL
   OR City IS NULL
   OR Address IS NULL
   OR TRIM(CustomerName) NOT RLIKE '^[A-Za-z ]+$'
   OR TRIM(Email) NOT RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'
   OR TRIM(City) NOT RLIKE '^[A-Za-z ]+$'
   OR LENGTH(TRIM(Address)) < 5;


In [0]:
CREATE OR REPLACE TABLE rejected_products
USING DELTA
AS
SELECT
    *,
    CASE
        WHEN ProductID IS NULL THEN 'Missing ProductID'
        WHEN ProductName IS NULL THEN 'Missing ProductName'
        WHEN Category IS NULL THEN 'Missing Category'
        WHEN UnitPrice IS NULL THEN 'Missing UnitPrice'
        WHEN CAST(UnitPrice AS DECIMAL(10,2)) <= 0
            THEN 'Invalid UnitPrice <= 0'
        WHEN TRIM(ProductName) NOT RLIKE '^[A-Za-z0-9 &-]+$'
            THEN 'Invalid ProductName Format'
        WHEN TRIM(Category) NOT RLIKE '^[A-Za-z &-]+$'
            THEN 'Invalid Category Format'
        ELSE 'Invalid Product Record'
    END AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.products_raw
WHERE ProductID IS NULL
   OR ProductName IS NULL
   OR Category IS NULL
   OR UnitPrice IS NULL
   OR CAST(UnitPrice AS DECIMAL(10,2)) <= 0
   OR TRIM(ProductName) NOT RLIKE '^[A-Za-z0-9 &-]+$'
   OR TRIM(Category) NOT RLIKE '^[A-Za-z &-]+$';

In [0]:
CREATE OR REPLACE TABLE rejected_stores
USING DELTA
AS
SELECT
    *,
    CASE
        WHEN StoreID IS NULL THEN 'Missing StoreID'
        WHEN StoreName IS NULL THEN 'Missing StoreName'
        WHEN Region IS NULL THEN 'Missing Region'
        WHEN TRIM(StoreName) NOT RLIKE '^[A-Za-z0-9 &-]+$'
            THEN 'Invalid StoreName Format'
        WHEN TRIM(Region) NOT RLIKE '^[A-Za-z ]+$'
            THEN 'Invalid Region Format'
        ELSE 'Invalid Store Record'
    END AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.stores_raw
WHERE StoreID IS NULL
   OR StoreName IS NULL
   OR Region IS NULL
   OR TRIM(StoreName) NOT RLIKE '^[A-Za-z0-9 &-]+$'
   OR TRIM(Region) NOT RLIKE '^[A-Za-z ]+$';

In [0]:
CREATE OR REPLACE TABLE rejected_sales
USING DELTA
AS
SELECT
    *,
    CASE
        WHEN TransactionID IS NULL THEN 'Missing TransactionID'
        WHEN CustomerID IS NULL THEN 'Missing CustomerID'
        WHEN ProductID IS NULL THEN 'Missing ProductID'
        WHEN StoreID IS NULL THEN 'Missing StoreID'
        WHEN Quantity IS NULL THEN 'Missing Quantity'
        WHEN CAST(Quantity AS INT) <= 0
            THEN 'Invalid Quantity <= 0'
        WHEN CAST(Quantity AS INT) > 1000
            THEN 'Quantity exceeds business limit'
        WHEN TO_DATE(TxnDate,'dd-MM-yyyy') IS NULL
            THEN 'Invalid Transaction Date'
        ELSE 'Invalid Sales Record'
    END AS RejectReason,
    CURRENT_TIMESTAMP() AS RejectTimestamp
FROM training_catalog.bronze.sales_raw
WHERE TransactionID IS NULL
   OR CustomerID IS NULL
   OR ProductID IS NULL
   OR StoreID IS NULL
   OR Quantity IS NULL
   OR CAST(Quantity AS INT) <= 0
   OR CAST(Quantity AS INT) > 1000
   OR TO_DATE(TxnDate,'dd-MM-yyyy') IS NULL;

In [0]:
SELECT 'customers' AS table_name, COUNT(*) AS rejected_rows FROM rejected_customers
UNION ALL
SELECT 'products', COUNT(*) FROM rejected_products
UNION ALL
SELECT 'stores', COUNT(*) FROM rejected_stores
UNION ALL
SELECT 'sales', COUNT(*) FROM rejected_sales;